# 01 — Data Exploration

Sanity-check the trained NextTrack artifacts: catalogue size, matrix sparsity,
tag coverage, and the noisiness of Last.fm tags (the known `why`-quality
limitation). Run `uv run train` first so `prototypes/artifacts/` exists.

In [9]:
import pickle
import numpy as np
from nextrack.train import FACTORS_PATH, IDMAP_PATH

factors = np.load(FACTORS_PATH)
maps = pickle.load(open(IDMAP_PATH, 'rb'))
names = maps['track_names']
tags = maps['tags']
print('track_factors shape:', factors.shape)
print('catalogue tracks   :', f"{len(maps['track_id_to_index']):,}")
print('tracks with names  :', f'{len(names):,}')
print('tracks with tags   :', f'{len(tags):,}')

track_factors shape: (35861, 64)
catalogue tracks   : 35,861
tracks with names  : 35,861
tracks with tags   : 24,668


## Tag coverage

How many recommendable tracks can actually carry a `why` explanation?

In [10]:
n_tracks = len(maps['track_id_to_index'])
n_tagged = sum(1 for tid in maps['track_id_to_index'] if tags.get(tid))
print(f'tag coverage: {n_tagged:,} / {n_tracks:,}  ({100*n_tagged/n_tracks:.1f}%)')
print('-> ~half of recommendations will have an empty why string (honest limitation).')

tag coverage: 24,668 / 35,861  (68.8%)
-> ~half of recommendations will have an empty why string (honest limitation).


## Top artists in the catalogue

Which artists dominate the trained slice (drives which seed sets demo well).

In [11]:
from collections import Counter
artist_counts = Counter(names[tid][0] for tid in maps['track_id_to_index'] if tid in names)
for artist, n in artist_counts.most_common(10):
    print(f'  {n:4}  {artist}')

   193  The Beatles
   114  Pink Floyd
   102  Disturbed
    91  Roxette
    90  Marilyn Manson
    89  Radiohead
    85  Lana Del Rey
    84  Metallica
    83  Muse
    80  David Bowie


## Tag noise — the `why`-quality limitation

Last.fm tags are free-text user contributions, not a clean genre vocabulary.
Some are genres (`classic rock`), some are band names (`Type O Negative`), some
are junk. We do NOT filter them — raw tags are honest about the data. A
production system would map to a controlled vocabulary (future work).

In [12]:
# show a few tracks whose tags mix genres with non-genre free text
shown = 0
for tid, tlist in tags.items():
    if shown >= 5:
        break
    if tid in names and len(tlist) >= 4:
        a, t = names[tid]
        print(f'{a} - {t}')
        print(f'    tags: {tlist[:6]}')
        shown += 1

Grouplove - Tongue Tied
    tags: ['indie', 'indie rock', 'indie pop', 'alternative', 'hipster indie', 'rock']
Interpol - The Heinrich Maneuver
    tags: ['indie rock', 'indie', 'alternative', 'rock', 'post-punk', 'interpol']
Santana - Oye Como Va
    tags: ['latin', 'rock', 'Latin Rock', 'santana', 'classic rock', 'guitar']
Elder - Legend
    tags: ['stoner metal', 'Stoner Rock', 'psychedelic', 'Progressive', 'heavy psych', 'played']
Greta Svabo Bech - Circles (based on Ludovico Einaudi "Experience")
    tags: ['amor amor', 'amor amor amor', 'Giusychevola e che ama', 'Greta Svabo Bech']
